## Inference mode for AST Model

In [ ]:
import os
import torch
import torchaudio
import torchaudio.transforms as T
from torch import nn
from torchvision import models
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "models/ast_model_synthetic.pth"
DATA_DIR = "data/trimmed_fan"
SAMPLE_RATE = 16000

# Preprocessing: resample and convert to log-Mel spectrogram
transform = nn.Sequential(
    T.Resample(orig_freq=44100, new_freq=SAMPLE_RATE),
    T.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=128),
    T.AmplitudeToDB()
)

class ASTModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.vit_b_16(weights="IMAGENET1K_V1")
        self.backbone.heads = nn.Linear(self.backbone.heads.in_features, 2)

    def forward(self, x):
        return self.backbone(x)

def load_audio(path):
    waveform, sr = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = T.Resample(sr, SAMPLE_RATE)(waveform)
    return waveform

def preprocess_audio(path):
    waveform = load_audio(path)
    spec = transform(waveform)  # [1, n_mels, time]
    spec = torch.nn.functional.interpolate(spec, size=(224, 224))  # Resize for ViT
    spec = (spec - spec.mean()) / spec.std()  # Normalize
    spec = spec.expand(3, -1, -1)  # AST expects 3-channel input
    return spec

def main():
    model = ASTModel().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    total = 0
    correct = 0

    label_map = {"normal": 0, "abnormal": 1}

    with torch.no_grad():
        for label_str, label_int in label_map.items():
            class_dir = os.path.join(DATA_DIR, label_str)
            for root, _, files in os.walk(class_dir):
                for file in tqdm(files, desc=f"Processing {label_str}"):
                    if not file.endswith(".wav"):
                        continue
                    file_path = os.path.join(root, file)
                    try:
                        input_tensor = preprocess_audio(file_path).unsqueeze(0).to(DEVICE)
                        output = model(input_tensor)
                        pred = torch.argmax(output, dim=1).item()
                        correct += int(pred == label_int)
                        total += 1
                    except Exception as e:
                        print(f"Error processing {file_path}: {e}")

    accuracy = 100.0 * correct / total if total > 0 else 0.0
    print(f"\n✅ Accuracy: {accuracy:.2f}% ({correct}/{total})")

if __name__ == "__main__":
    main()
